# Week1_ex2 - Force Sweep on a C-Core Electromagnet

This exercise models a C-core actuator made of a steel core, a coil, and a bias magnet, pulling on a separate steel bar (the armature). The coil current is swept from 0 A to 500 A in 100 A steps, and the force on the bar is calculated at each step using the Magnetostatic solver in Maxwell 3D.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time

In [ ]:
## Initialize AEDT Desktop session and create Maxwell 3D project/design ##

# Start AEDT session (specify version and student license flag)
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# Disable autosave to prevent popup interruptions during scripted runs
DT.disable_autosave()

# Solution type: Magnetostatic
sol_type = "Magnetostatic"

# Create Maxwell 3D design object
# On first creation, a new project and design are generated with default names
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type, student_version=True)
# Reference to odesign, used for AEDT's built-in recording-style API calls
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##

# Project name (change freely as needed)
proj_name = "Week1_ex2"

# Compute save directory path
dir = os.getcwd() + f"\\{proj_name}"
print(dir)

# # Create directory (delete any existing folder with the same name before re-running)
os.makedirs(dir, exist_ok=True)

# Design name
desi_name = "Week1_ex2"


In [ ]:
## Save project and apply design name ##

# Reference to the project object containing this design
proj = M3D.oproject

# # Save project
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)

# Rename design to the desired name
M3D.rename_design(desi_name, save=False)
M3D.save_project()

In [ ]:
## Create geometry ##

# Build the C-core

origin = [0, 0, -5]
sizes = [10, -30, 10]
box1 = M3D.modeler.create_box(origin=origin, sizes=sizes, name=None, material=None)

vector = [30, 0, 0]
M3D.modeler.duplicate_along_line(assignment=box1, vector=vector, clones=2, attach=False, is_3d_comp=False, duplicate_assignment=True)
box2 = M3D.modeler.object_list[-1]

origin = [0, -30, -5]
sizes = [50, -10, 10]
box3 = M3D.modeler.create_box(origin=origin, sizes=sizes, name=None, material=None)

core = M3D.modeler.unite(assignment=[box1, box2, box3], purge=False, keep_originals=False)
core = M3D.modeler.get_object_from_name(assignment=core)    # unite() returns only the object name, so re-fetch the actual object reference

origin = [0, 0, 0]
vector = [0, 1, 0]
M3D.modeler.duplicate_and_mirror(assignment=core, origin=origin, vector=vector, is_3d_comp=False, duplicate_assignment=True)
core_tmp = M3D.modeler.object_list[-1]  # duplicate_and_mirror() also returns only the object name, so re-fetch the object reference

M3D.modeler.unite(assignment=[core, core_tmp], purge=False, keep_originals=False)

M3D.assign_material(assignment=core, material="steel_1008")

core.name = "Core"


# Create the moving bar (armature)

origin = [51, -40, -5]
sizes = [10, 80, 10]
bar = M3D.modeler.create_box(origin=origin, sizes=sizes, name="Bar", material="steel_1008")


# Create the coil

origin = [45, 30, 10]
sizes = [-20, -60, -20]
coil = M3D.modeler.create_box(origin=origin, sizes=sizes, name="Coil", material="copper")

coil.subtract(tool_list=core, keep_originals=True)


# Create the permanent magnet

origin = [0, -10, -5]
sizes = [10, 20, 10]
magnet = M3D.modeler.create_box(origin=origin, sizes=sizes, name="Magnet", material="NdFe35")

core.subtract(tool_list=magnet, keep_originals=True)

In [ ]:
## Set magnet polarization direction ##

# Create a coordinate system attached to the magnet
origin = magnet.faces[0].center
M3D.modeler.create_coordinate_system(origin=origin, reference_cs='Global', name="FaceCS1", mode='axis', view='iso', x_pointing=None, y_pointing=None, psi=0, theta=0, phi=0, u=None)

# Reassign the magnet's part coordinate system
magnet.part_coordinate_system = "FaceCS1"
M3D.modeler.set_working_coordinate_system("Global") # Return to the Global coordinate system

# Check and update the magnet's magnetization direction
NdFe35 = M3D.materials.exists_material(material="NdFe35")
print(NdFe35.get_magnetic_coercivity())

NdFe35.set_magnetic_coercivity('-890000', x=0, y=1, z=0)
print(NdFe35.get_magnetic_coercivity())


In [ ]:
## Assign insulating boundary to the coil ##

coil_boundary = M3D.assign_insulating(assignment=coil, insulation=None)

In [ ]:
## Apply current excitation to the coil ##

# Create the coil terminal (current cross-section)

M3D.modeler.section(assignment=coil, plane="XY", create_new=True, section_cross_object=False)
coil_section = M3D.modeler.sheet_objects[-1]    # Select the most recently created section object

coil_terminal = M3D.modeler.separate_bodies(assignment=coil_section, create_group=False)     

M3D.modeler.delete(assignment= (M3D.modeler.sheet_objects[-1]) )    # Remove the now-unneeded section object
coil_terminal[0].name = "Coil_Terminal"

# Excitation

M3D["c1"] = "100A"  # Add design variable
current1 = M3D.assign_current(assignment=coil_terminal, amplitude="c1", phase='0deg', solid=False, swap_direction=False, name="Current1")


In [ ]:
## Assign virtual force boundary condition to the bar ##

M3D.assign_force(assignment=bar, coordinate_system='Global', is_virtual=True, force_name="Force1")


In [ ]:
## Create simulation region (surrounding air/vacuum domain) ##

region = M3D.modeler.create_region(pad_value=50, pad_type='Percentage Offset', name='Region')

In [ ]:
## Create and inspect analysis setup ##

# Create analysis setup object
my_setup = M3D.create_setup(name="Setup1")

# Check available setup properties for the current solution type (stored as a dict)
display(my_setup.props)

In [ ]:
# Modify desired properties from the dict above (maximum number of adaptive passes)

my_setup.props['MaximumPasses'] = 10


In [ ]:
# Run the analysis

my_setup.analyze()

In [ ]:
## Set up parametric sweep of coil current ##

param_Setups = M3D.parametrics    # Reference to the parametrics setup collection

c1_param = param_Setups.add(variable="c1", start_point="0A", end_point="500A", step="100A", variation_type='LinearStep', solution=None, name="c1_param")

# Check sweep setup properties
c1_param.props

In [ ]:
# Enable field saving so results are accessible at every sweep point
c1_param.props['ProdOptiSetupDataV2'] = {'SaveFields': True,
                                            'CopyMesh': True,
                                            'SolveWithCopiedMeshOnly': True}

# Confirm the change
c1_param.props

In [ ]:
# Run the parametric sweep

c1_param.analyze()

In [ ]:
# Save final results

M3D.save_project()